In [1]:
!sudo apt-get update && sudo apt-get upgrade && sudo apt-get install cmake
!pip install https://github.com/kpu/kenlm/archive/master.zip
!pip install levenshtein
!pip install torch torchaudio transformers

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,243 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [2,788 kB]
Hit:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [8,82

In [6]:
from typing import List, Tuple
import heapq
import kenlm
import torch
import torchaudio
import Levenshtein
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC


In [ ]:


class Wav2Vec2Decoder:
    def __init__(
            self,
            model_name="facebook/wav2vec2-base-960h",
            lm_model_path="/content/3-gram.pruned.1e-7.arpa",
            beam_width=3,
            alpha=1.0,
            beta=1.0
        ):
        """
        Initialization of Wav2Vec2Decoder class

        Args:
            model_name (str): Pretrained Wav2Vec2 model from transformers
            lm_model_path (str): Path to the KenLM n-gram model (for LM rescoring)
            beam_width (int): Number of hypotheses to keep in beam search
            alpha (float): LM weight for shallow fusion and rescoring
            beta (float): Word bonus for shallow fusion
        """
        self.processor = Wav2Vec2Processor.from_pretrained(model_name)
        self.model = Wav2Vec2ForCTC.from_pretrained(model_name)

        # Словарь модели
        self.vocab = {i: c for c, i in self.processor.tokenizer.get_vocab().items()}
        self.blank_token_id = self.processor.tokenizer.pad_token_id
        self.word_delimiter = self.processor.tokenizer.word_delimiter_token
        self.beam_width = beam_width
        self.alpha = alpha
        self.beta = beta
        self.lm_model = kenlm.Model(lm_model_path) if lm_model_path else None

    def greedy_decode(self, logits: torch.Tensor) -> str:
        """
        Perform greedy decoding (find best CTC path)

        Args:
            logits (torch.Tensor): Logits from Wav2Vec2 model (T, V)

        Returns:
            str: Decoded transcript
        """
        # Находим индекс максимального логита для каждого шага (T - количество временных шагов)
        pred_ids = torch.argmax(logits, dim=-1)  # Выбираем максимальные значения по оси vocab (V)

        # Преобразуем индексы в символы, игнорируя пустой токен <pad>
        pred_tokens = [self.vocab[idx.item()] for idx in pred_ids if idx.item() != self.blank_token_id]

        # Собираем строку из символов
        decoded_string = ''.join(pred_tokens)
        return decoded_string


    def decode(self, audio_input: torch.Tensor, method: str = "greedy") -> str:
        """
        Decode input audio file using the specified method

        Args:
            audio_input (torch.Tensor): Audio tensor
            method (str): Decoding method ("greedy", "beam", "beam_lm", "beam_lm_rescore"),
                where "greedy" is a greedy decoding,
                      "beam" is beam search without LM,
                      "beam_lm" is beam search with LM shallow fusion, and
                      "beam_lm_rescore" is a beam search with second pass LM rescoring

        Returns:
            str: Decoded transcription
        """
        # Преобразуем аудиофайл в формат, который подходит для модели
        inputs = self.processor(audio_input, return_tensors="pt", sampling_rate=16000)
        with torch.no_grad():
            logits = self.model(inputs.input_values.squeeze(0)).logits[0]

        # В зависимости от метода декодирования вызываем соответствующую функцию
        if method == "greedy":
            return self.greedy_decode(logits)
       # elif method == "beam":
           # return self.beam_search_decode(logits)
       # elif method == "beam_lm":
          #  return self.beam_search_with_lm(logits)
        #elif method == "beam_lm_rescore":
          #  beams = self.beam_search_decode(logits, return_beams=True)
          #  return self.lm_rescore(beams)
       # else:
          #  raise ValueError("Invalid decoding method. Choose one of 'greedy', 'beam', 'beam_lm', 'beam_lm_rescore'.")




In [ ]:
# Тестирование декодера
def test(decoder, audio_path, true_transcription):
    import Levenshtein

    # Загружаем аудиофайл и проверяем частоту дискретизации
    audio_input, sr = torchaudio.load(audio_path)
    assert sr == 16000, "Sample rate must be 16kHz"

    print("=" * 60)
    print("Target transcription")
    print(true_transcription)

    # Тестируем все методы декодирования
    for d_strategy in ["greedy"]: #"beam", "beam_lm", "beam_lm_rescore"]:
        print("-" * 60)
        print(f"{d_strategy} decoding")
        transcript = decoder.decode(audio_input, method=d_strategy)
        print(f"{transcript}")
        print(f"Character-level Levenshtein distance: {Levenshtein.distance(true_transcription, transcript.strip())}")


if __name__ == "__main__":
    # Пример тестовых аудиофайлов и их транскрипций
    test_samples = [
        ("/content/assignments_assignment2_examples_sample1.wav", "IF YOU ARE GENEROUS HERE IS A FITTING OPPORTUNITY FOR THE EXERCISE OF YOUR MAGNANIMITY IF YOU ARE PROUD HERE AM I YOUR RIVAL READY TO ACKNOWLEDGE MYSELF YOUR DEBTOR FOR AN ACT OF THE MOST NOBLE FORBEARANCE"),
        ("/content/sample2.wav", "AND IF ANY OF THE OTHER COPS HAD PRIVATE RACKETS OF THEIR OWN IZZY WAS UNDOUBTEDLY THE MAN TO FIND IT OUT AND USE THE INFORMATION WITH A BEAT SUCH AS THAT EVEN GOING HALVES AND WITH ALL THE GRAFT TO THE UPPER BRACKETS HE'D STILL BE ABLE TO MAKE HIS PILE IN A MATTER OF MONTHS"),
        ("/content/sample3.wav", "GUESS A MAN GETS USED TO ANYTHING HELL MAYBE I CAN HIRE SOME BUMS TO SIT AROUND AND WHOOP IT UP WHEN THE SHIPS COME IN AND BILL THIS AS A REAL OLD MARTIAN DEN OF SIN"),
        ("/content/sample4.wav", "IT WAS A TUNE THEY HAD ALL HEARD HUNDREDS OF TIMES SO THERE WAS NO DIFFICULTY IN TURNING OUT A PASSABLE IMITATION OF IT TO THE IMPROVISED STRAINS OF I DIDN'T WANT TO DO IT THE PRISONER STRODE FORTH TO FREEDOM"),
        ("/content/sample5.wav", "MARGUERITE TIRED OUT WITH THIS LONG CONFESSION THREW HERSELF BACK ON THE SOFA AND TO STIFLE A SLIGHT COUGH PUT UP HER HANDKERCHIEF TO HER LIPS AND FROM THAT TO HER EYES"),
        ("/content/sample6.wav", "AT THIS TIME ALL PARTICIPANTS ARE IN A LISTEN ONLY MODE"),
        ("/content/sample7.wav", "THE INCREASE WAS MAINLY ATTRIBUTABLE TO THE NET INCREASE IN THE AVERAGE SIZE OF OUR FLEETS"),
        ("/content/sample8.wav", "OPERATING SURPLUS IS A NON CAP FINANCIAL MEASURE WHICH IS DEFINED AS FULLY IN OUR PRESS RELEASE"),
    ]

    # Создаем объект декодера
    decoder = Wav2Vec2Decoder()

    # Тестируем все образцы
    _ = [test(decoder, audio_path, target) for audio_path, target in test_samples]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.60k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Target transcription
IF YOU ARE GENEROUS HERE IS A FITTING OPPORTUNITY FOR THE EXERCISE OF YOUR MAGNANIMITY IF YOU ARE PROUD HERE AM I YOUR RIVAL READY TO ACKNOWLEDGE MYSELF YOUR DEBTOR FOR AN ACT OF THE MOST NOBLE FORBEARANCE
------------------------------------------------------------
greedy decoding
IF|YOOU|AARE||||GENNERROUSS||HHEREE|ISS||||A|FITTTINGG||OPPPOORRTTUNNITTY||FOOR||||THEE||EXERRCISSE||OF||YOURR||MMAGNNANNIMITY|||IFF||YOOU||ARE|||PRROUD||||HEERRE|AMM|||II|||YOUR||RIVALL|||RRETER|TO|ACKKNNOWLEDDGEE|||MYSSELLF|||||YOURR|||DEPTTORR|||FOR|||AN|||ACCT||OF|||||||MMOSTT|||NOBLLE||FORBEAARRANCE||
Character-level Levenshtein distance: 146
Target transcription
AND IF ANY OF THE OTHER COPS HAD PRIVATE RACKETS OF THEIR OWN IZZY WAS UNDOUBTEDLY THE MAN TO FIND IT OUT AND USE THE INFORMATION WITH A BEAT SUCH AS THAT EVEN GOING HALVES AND WITH ALL THE GRAFT TO THE UPPER BRACKETS HE'D STILL BE ABLE TO MAKE HIS PILE IN A MATTER OF MONTHS
-------------------------------------------------

In [ ]:
#Метод Beam_search_decode

In [ ]:
import heapq
import torch
from typing import List, Tuple

class Wav2Vec2Decoder:
    def __init__(
            self,
            model_name="facebook/wav2vec2-base-960h",
            lm_model_path="/content/3-gram.pruned.1e-7.arpa",
            beam_width=3,
            alpha=1.0,
            beta=1.0
        ):
        """
        Initialization of Wav2Vec2Decoder class

        Args:
            model_name (str): Pretrained Wav2Vec2 model from transformers
            lm_model_path (str): Path to the KenLM n-gram model (for LM rescoring)
            beam_width (int): Number of hypotheses to keep in beam search
            alpha (float): LM weight for shallow fusion and rescoring
            beta (float): Word bonus for shallow fusion
        """
        self.processor = Wav2Vec2Processor.from_pretrained(model_name)
        self.model = Wav2Vec2ForCTC.from_pretrained(model_name)

        # Словарь модели
        self.vocab = {i: c for c, i in self.processor.tokenizer.get_vocab().items()}
        self.blank_token_id = self.processor.tokenizer.pad_token_id
        self.word_delimiter = self.processor.tokenizer.word_delimiter_token
        self.beam_width = beam_width
        self.alpha = alpha
        self.beta = beta
        self.lm_model = kenlm.Model(lm_model_path) if lm_model_path else None

    def beam_search_decode(self, logits: torch.Tensor, return_beams: bool = False):
        """
        Perform beam search decoding (no LM)

        Args:
            logits (torch.Tensor): Logits from Wav2Vec2 model (T, V), where
                T - number of time steps and
                V - vocabulary size
            return_beams (bool): Return all beam hypotheses for second pass LM rescoring

        Returns:
            Union[str, List[Tuple[float, List[int]]]]:
                (str) - If return_beams is False, returns the best decoded transcript as a string.
                (List[Tuple[List[int], float]]) - If return_beams is True, returns a list of tuples
                    containing hypotheses and log probabilities.
        """
        # Применяем log_softmax для логарифмированных вероятностей
        log_probs = torch.log_softmax(logits, dim=-1)

        # Инициализируем начальную гипотезу: пустая последовательность и вероятность 0
        beams = [([], 0.0)]  # (hypothesis, log_prob)

        # Проходим по каждому времени (T)
        for t in range(log_probs.size(0)):
            all_candidates = []
            # Проходим по текущим гипотезам в лучах
            for hypothesis, score in beams:
                for vocab_idx in range(log_probs.size(1)):
                    token_prob = log_probs[t, vocab_idx]
                    if vocab_idx == self.blank_token_id:
                        # Пропускаем пустой токен
                        continue
                    candidate = (hypothesis + [vocab_idx], score + token_prob.item())
                    all_candidates.append(candidate)

            # Сортируем все кандидаты по вероятности (score) и выбираем top beam_width кандидатов
            beams = heapq.nlargest(self.beam_width, all_candidates, key=lambda x: x[1])

        # Получаем лучшую гипотезу (последний элемент)
        best_hypothesis, _ = beams[0]

        # Преобразуем индексы токенов в строку
        decoded_string = ''.join([self.vocab[idx] for idx in best_hypothesis])

        if return_beams:
            return beams  # Возвращаем все лучи для дальнейшего переоценивания с LM
        else:
            return decoded_string  # Возвращаем лучшую гипотезу

    def decode(self, audio_input: torch.Tensor, method: str = "beam") -> str:
        """
        Decode input audio file using the specified method

        Args:
            audio_input (torch.Tensor): Audio tensor
            method (str): Decoding method ("greedy", "beam", "beam_lm", "beam_lm_rescore"),
                where "greedy" is a greedy decoding,
                      "beam" is beam search without LM,
                      "beam_lm" is beam search with LM shallow fusion, and
                      "beam_lm_rescore" is a beam search with second pass LM rescoring

        Returns:
            str: Decoded transcription
        """
        # Преобразуем аудиофайл в формат, который подходит для модели
        inputs = self.processor(audio_input, return_tensors="pt", sampling_rate=16000)
        with torch.no_grad():
            logits = self.model(inputs.input_values.squeeze(0)).logits[0]

        # В зависимости от метода декодирования вызываем соответствующую функцию
        if method == "greedy":
            return self.greedy_decode(logits)
        elif method == "beam":
            return self.beam_search_decode(logits)
       # elif method == "beam_lm":
          #  return self.beam_search_with_lm(logits)
        #elif method == "beam_lm_rescore":
          #  beams = self.beam_search_decode(logits, return_beams=True)
          #  return self.lm_rescore(beams)
       # else:
          #  raise ValueError("Invalid decoding method. Choose one of 'greedy', 'beam', 'beam_lm', 'beam_lm_rescore'.")





In [ ]:
# Тестирование декодера
def test(decoder, audio_path, true_transcription):
    import Levenshtein

    # Загружаем аудиофайл и проверяем частоту дискретизации
    audio_input, sr = torchaudio.load(audio_path)
    assert sr == 16000, "Sample rate must be 16kHz"

    print("=" * 60)
    print("Target transcription")
    print(true_transcription)

    # Тестируем все методы декодирования
    for d_strategy in ["beam"]: #"beam", "beam_lm", "beam_lm_rescore"]:
        print("-" * 60)
        print(f"{d_strategy} decoding")
        transcript = decoder.decode(audio_input, method=d_strategy)
        print(f"{transcript}")
        print(f"Character-level Levenshtein distance: {Levenshtein.distance(true_transcription, transcript.strip())}")


if __name__ == "__main__":
    # Пример тестовых аудиофайлов и их транскрипций
    test_samples = [
        ("/content/assignments_assignment2_examples_sample1.wav", "IF YOU ARE GENEROUS HERE IS A FITTING OPPORTUNITY FOR THE EXERCISE OF YOUR MAGNANIMITY IF YOU ARE PROUD HERE AM I YOUR RIVAL READY TO ACKNOWLEDGE MYSELF YOUR DEBTOR FOR AN ACT OF THE MOST NOBLE FORBEARANCE"),
        ("/content/sample2.wav", "AND IF ANY OF THE OTHER COPS HAD PRIVATE RACKETS OF THEIR OWN IZZY WAS UNDOUBTEDLY THE MAN TO FIND IT OUT AND USE THE INFORMATION WITH A BEAT SUCH AS THAT EVEN GOING HALVES AND WITH ALL THE GRAFT TO THE UPPER BRACKETS HE'D STILL BE ABLE TO MAKE HIS PILE IN A MATTER OF MONTHS"),
        ("/content/sample3.wav", "GUESS A MAN GETS USED TO ANYTHING HELL MAYBE I CAN HIRE SOME BUMS TO SIT AROUND AND WHOOP IT UP WHEN THE SHIPS COME IN AND BILL THIS AS A REAL OLD MARTIAN DEN OF SIN"),
        ("/content/sample4.wav", "IT WAS A TUNE THEY HAD ALL HEARD HUNDREDS OF TIMES SO THERE WAS NO DIFFICULTY IN TURNING OUT A PASSABLE IMITATION OF IT TO THE IMPROVISED STRAINS OF I DIDN'T WANT TO DO IT THE PRISONER STRODE FORTH TO FREEDOM"),
        ("/content/sample5.wav", "MARGUERITE TIRED OUT WITH THIS LONG CONFESSION THREW HERSELF BACK ON THE SOFA AND TO STIFLE A SLIGHT COUGH PUT UP HER HANDKERCHIEF TO HER LIPS AND FROM THAT TO HER EYES"),
        ("/content/sample6.wav", "AT THIS TIME ALL PARTICIPANTS ARE IN A LISTEN ONLY MODE"),
        ("/content/sample7.wav", "THE INCREASE WAS MAINLY ATTRIBUTABLE TO THE NET INCREASE IN THE AVERAGE SIZE OF OUR FLEETS"),
        ("/content/sample8.wav", "OPERATING SURPLUS IS A NON CAP FINANCIAL MEASURE WHICH IS DEFINED AS FULLY IN OUR PRESS RELEASE"),
    ]

    # Создаем объект декодера
    decoder = Wav2Vec2Decoder()

    # Тестируем все образцы
    _ = [test(decoder, audio_path, target) for audio_path, target in test_samples]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Target transcription
IF YOU ARE GENEROUS HERE IS A FITTING OPPORTUNITY FOR THE EXERCISE OF YOUR MAGNANIMITY IF YOU ARE PROUD HERE AM I YOUR RIVAL READY TO ACKNOWLEDGE MYSELF YOUR DEBTOR FOR AN ACT OF THE MOST NOBLE FORBEARANCE
------------------------------------------------------------
beam decoding
NIWWWWWANNWNNINHIIF|||YOOU||WAARE|||||GGHHRRREEENNNEEERRRIALAOUUSSS||||NNNNNWHHHEEREE||IISS||||A||FFEEEITTTTTTIINGG||OOP||PPROORRITTTRRRUUUNNNNIIITTTTRYYY|||FFROOR||||THEE|||NNEEEXXTTTHHERRR||CHLLRRLIIISSSEE|||NNNNNNNNNNNNNNNBHOF|||YYOOURR|||MMMM|AGG|HHNNNAANNNNNNNNIM|MMMIITTTTTTEERRYN||||NNUUUUUUUUUUUUWWUWNWNNNU|UUNIHIIFF||YOOU||WAAREE|||||PPRRRAARRROOUNDD||||||UUNNNNWIIIIIIUIWHHHEEERRE|||HAAMMM|||||HHHHHHHIIIL|||||IIIIUNUIUUNWUNNUUUIYYYYOURR|||RRR|AIIIEVVVVAALLL||||UUUUUWWUWWIWIIIIIWWWRRRHEAD|THHLER||THROU||AC|KKKNNNOWWLLEDDGEE|||MMYYY||||SSSSSU|EEELLFFF|||||TYHOOU|ARR||||||AASSDHOO|EUUPPIIITTTHHHHORRR||||UWUUU||OFFRORR|||AN||||HHRRHACCITT||||OF|||||||MMMOOSSSTT|||SSNKNNNNOOOABBLLEE|||FF

In [ ]:
Beam_search with LM

In [ ]:
import heapq
import torch
import kenlm
from typing import List, Tuple

class Wav2Vec2Decoder:
    def __init__(
            self,
            model_name="facebook/wav2vec2-base-960h",
            lm_model_path="/content/3-gram.pruned.1e-7.arpa",
            beam_width=3,
            alpha=1.0,
            beta=1.0
        ):
        """
        Initialization of Wav2Vec2Decoder class

        Args:
            model_name (str): Pretrained Wav2Vec2 model from transformers
            lm_model_path (str): Path to the KenLM n-gram model (for LM rescoring)
            beam_width (int): Number of hypotheses to keep in beam search
            alpha (float): LM weight for shallow fusion and rescoring
            beta (float): Word bonus for shallow fusion
        """
        self.processor = Wav2Vec2Processor.from_pretrained(model_name)
        self.model = Wav2Vec2ForCTC.from_pretrained(model_name)

        # Словарь модели
        self.vocab = {i: c for c, i in self.processor.tokenizer.get_vocab().items()}
        self.blank_token_id = self.processor.tokenizer.pad_token_id
        self.word_delimiter = self.processor.tokenizer.word_delimiter_token
        self.beam_width = beam_width
        self.alpha = alpha
        self.beta = beta
        self.lm_model = kenlm.Model(lm_model_path) if lm_model_path else None

    def beam_search_with_lm(self, logits: torch.Tensor) -> str:
        """
        Perform beam search decoding with shallow LM fusion

        Args:
            logits (torch.Tensor): Logits from Wav2Vec2 model (T, V), where
                T - number of time steps and
                V - vocabulary size

        Returns:
            str: Decoded transcript
        """
        if not self.lm_model:
            raise ValueError("KenLM model required for LM shallow fusion")

        # Применяем log_softmax для логарифмированных вероятностей
        log_probs = torch.log_softmax(logits, dim=-1)

        # Инициализируем начальную гипотезу: пустая последовательность и вероятность 0
        beams = [([], 0.0)]  # (hypothesis, log_prob)

        # Проходим по каждому времени (T)
        for t in range(log_probs.size(0)):
            all_candidates = []
            # Проходим по текущим гипотезам в лучах
            for hypothesis, score in beams:
                for vocab_idx in range(log_probs.size(1)):
                    token_prob = log_probs[t, vocab_idx]
                    if vocab_idx == self.blank_token_id:
                        # Пропускаем пустой токен
                        continue

                    # Получаем строковое представление токенов в гипотезе
                    word = self.vocab[vocab_idx]
                    hypothesis_words = [self.vocab[idx] for idx in hypothesis]
                    sentence = ' '.join(hypothesis_words + [word])

                    # Получаем вероятность для токена от модели языка
                    lm_prob = self.lm_model.score(sentence) if hypothesis else 0.0

                    # Применяем поверхностную интеграцию (суммируем log probs и LM)
                    combined_score = score + token_prob.item() + self.alpha * lm_prob

                    candidate = (hypothesis + [vocab_idx], combined_score)
                    all_candidates.append(candidate)

            # Сортируем все кандидаты по вероятности (score) и выбираем top beam_width кандидатов
            beams = heapq.nlargest(self.beam_width, all_candidates, key=lambda x: x[1])

        # Получаем лучшую гипотезу (последний элемент)
        best_hypothesis, _ = beams[0]

        # Преобразуем индексы токенов в строку
        decoded_string = ''.join([self.vocab[idx] for idx in best_hypothesis])

        return decoded_string


    def decode(self, audio_input: torch.Tensor, method: str = "beam_lm") -> str:
        """
        Decode input audio file using the specified method

        Args:
            audio_input (torch.Tensor): Audio tensor
            method (str): Decoding method ("greedy", "beam", "beam_lm", "beam_lm_rescore"),
                where "greedy" is a greedy decoding,
                      "beam" is beam search without LM,
                      "beam_lm" is beam search with LM shallow fusion, and
                      "beam_lm_rescore" is a beam search with second pass LM rescoring

        Returns:
            str: Decoded transcription
        """
        # Преобразуем аудиофайл в формат, который подходит для модели
        inputs = self.processor(audio_input, return_tensors="pt", sampling_rate=16000)
        with torch.no_grad():
            logits = self.model(inputs.input_values.squeeze(0)).logits[0]

        # В зависимости от метода декодирования вызываем соответствующую функцию
        if method == "greedy":
            return self.greedy_decode(logits)
        elif method == "beam":
            return self.beam_search_decode(logits)
        elif method == "beam_lm":
            return self.beam_search_with_lm(logits)
        #elif method == "beam_lm_rescore":
          #  beams = self.beam_search_decode(logits, return_beams=True)
          #  return self.lm_rescore(beams)
       # else:
          #  raise ValueError("Invalid decoding method. Choose one of 'greedy', 'beam', 'beam_lm', 'beam_lm_rescore'.")





In [ ]:
# Тестирование декодера
def test(decoder, audio_path, true_transcription):
    import Levenshtein

    # Загружаем аудиофайл и проверяем частоту дискретизации
    audio_input, sr = torchaudio.load(audio_path)
    assert sr == 16000, "Sample rate must be 16kHz"

    print("=" * 60)
    print("Target transcription")
    print(true_transcription)

    # Тестируем все методы декодирования
    for d_strategy in ["beam_lm"]: #"beam", "beam_lm", "beam_lm_rescore"]:
        print("-" * 60)
        print(f"{d_strategy} decoding")
        transcript = decoder.decode(audio_input, method=d_strategy)
        print(f"{transcript}")
        print(f"Character-level Levenshtein distance: {Levenshtein.distance(true_transcription, transcript.strip())}")


if __name__ == "__main__":
    # Пример тестовых аудиофайлов и их транскрипций
    test_samples = [
        ("/content/assignments_assignment2_examples_sample1.wav", "IF YOU ARE GENEROUS HERE IS A FITTING OPPORTUNITY FOR THE EXERCISE OF YOUR MAGNANIMITY IF YOU ARE PROUD HERE AM I YOUR RIVAL READY TO ACKNOWLEDGE MYSELF YOUR DEBTOR FOR AN ACT OF THE MOST NOBLE FORBEARANCE"),
        ("/content/sample2.wav", "AND IF ANY OF THE OTHER COPS HAD PRIVATE RACKETS OF THEIR OWN IZZY WAS UNDOUBTEDLY THE MAN TO FIND IT OUT AND USE THE INFORMATION WITH A BEAT SUCH AS THAT EVEN GOING HALVES AND WITH ALL THE GRAFT TO THE UPPER BRACKETS HE'D STILL BE ABLE TO MAKE HIS PILE IN A MATTER OF MONTHS"),
        ("/content/sample3.wav", "GUESS A MAN GETS USED TO ANYTHING HELL MAYBE I CAN HIRE SOME BUMS TO SIT AROUND AND WHOOP IT UP WHEN THE SHIPS COME IN AND BILL THIS AS A REAL OLD MARTIAN DEN OF SIN"),
        ("/content/sample4.wav", "IT WAS A TUNE THEY HAD ALL HEARD HUNDREDS OF TIMES SO THERE WAS NO DIFFICULTY IN TURNING OUT A PASSABLE IMITATION OF IT TO THE IMPROVISED STRAINS OF I DIDN'T WANT TO DO IT THE PRISONER STRODE FORTH TO FREEDOM"),
        ("/content/sample5.wav", "MARGUERITE TIRED OUT WITH THIS LONG CONFESSION THREW HERSELF BACK ON THE SOFA AND TO STIFLE A SLIGHT COUGH PUT UP HER HANDKERCHIEF TO HER LIPS AND FROM THAT TO HER EYES"),
        ("/content/sample6.wav", "AT THIS TIME ALL PARTICIPANTS ARE IN A LISTEN ONLY MODE"),
        ("/content/sample7.wav", "THE INCREASE WAS MAINLY ATTRIBUTABLE TO THE NET INCREASE IN THE AVERAGE SIZE OF OUR FLEETS"),
        ("/content/sample8.wav", "OPERATING SURPLUS IS A NON CAP FINANCIAL MEASURE WHICH IS DEFINED AS FULLY IN OUR PRESS RELEASE"),
    ]

    # Создаем объект декодера
    decoder = Wav2Vec2Decoder()

    # Тестируем все образцы
    _ = [test(decoder, audio_path, target) for audio_path, target in test_samples]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Target transcription
IF YOU ARE GENEROUS HERE IS A FITTING OPPORTUNITY FOR THE EXERCISE OF YOUR MAGNANIMITY IF YOU ARE PROUD HERE AM I YOUR RIVAL READY TO ACKNOWLEDGE MYSELF YOUR DEBTOR FOR AN ACT OF THE MOST NOBLE FORBEARANCE
------------------------------------------------------------
beam_lm decoding
IIIIIIIIIIIIIIIIIIF|||YOUU|||AARE|||||GGERRRREEENNNEEERRRRAAAALSSSS||||||||||HHHEEREE|||ISS|||||||FFFERITTTTTTHINGG||OOPE|PPOOORRATTTRRRRUUNNNNIIIITTTRYYY|||FFFOOR||||THEE||||NEEEXXVTHHHERRRRCCCCCLLLIIIISSEE||||||||||||||||||||OT|||YYYOURR||||MMMMAGGGGHNNNNANNNNNNNNIMMMMMMITTTTTTERERYN||||||||||||||||||||||||||||||||||IIFF||YOOU|||ARRRE||||||PRRRRRRRRROULLD||||||||||||||||||||||HHHEEERRE|||||AMMM|||||||HHHHAIII||||||||||||||||||||||||||YYYOURR|||RRRRRIIIIVIIIAALL||||||||||||||||||||||WWWWRRHEADETHHHER|||||E||||||||KNNNOWLLLEEDGEE|||MMMYY|||||SSSSSEEEELLFFE||||||YYYOU||RR||||||||||DHOOOEEEPPPPITTTHHHHERRR||||||||||||FFFORR|||A||||||ELLLACCHTT||||OF|||||||MMMMOSSSTT||||ESNNNNOOOOBBBLLEE||

In [ ]:
import heapq
import torch
import kenlm
from typing import List, Tuple

class Wav2Vec2Decoder:
    def __init__(
            self,
            model_name="facebook/wav2vec2-base-960h",
            lm_model_path="/content/3-gram.pruned.1e-7.arpa",
            beam_width=3,
            alpha=1.0,
            beta=1.0
    ):
        """
        Initialization of Wav2Vec2Decoder class

        Args:
            model_name (str): Pretrained Wav2Vec2 model from transformers
            lm_model_path (str): Path to the KenLM n-gram model (for LM rescoring)
            beam_width (int): Number of hypotheses to keep in beam search
            alpha (float): LM weight for shallow fusion and rescoring
            beta (float): Word bonus for shallow fusion
        """
        self.processor = Wav2Vec2Processor.from_pretrained(model_name)
        self.model = Wav2Vec2ForCTC.from_pretrained(model_name)

        # Словарь модели
        self.vocab = {i: c for c, i in self.processor.tokenizer.get_vocab().items()}
        self.blank_token_id = self.processor.tokenizer.pad_token_id
        self.word_delimiter = self.processor.tokenizer.word_delimiter_token
        self.beam_width = beam_width
        self.alpha = alpha
        self.beta = beta
        self.lm_model = kenlm.Model(lm_model_path) if lm_model_path else None


    def beam_search_decode(self, logits: torch.Tensor, return_beams: bool = False):
        """
        Perform beam search decoding (no LM)

        Args:
            logits (torch.Tensor): Logits from Wav2Vec2 model (T, V), where
                T - number of time steps and
                V - vocabulary size
            return_beams (bool): Return all beam hypotheses for second pass LM rescoring

        Returns:
            Union[str, List[Tuple[float, List[int]]]]:
                (str) - If return_beams is False, returns the best decoded transcript as a string.
                (List[Tuple[List[int], float]]) - If return_beams is True, returns a list of tuples
                    containing hypotheses and log probabilities.
        """
        # Применяем log_softmax для логарифмированных вероятностей
        log_probs = torch.log_softmax(logits, dim=-1)

        # Инициализируем начальную гипотезу: пустая последовательность и вероятность 0
        beams = [([], 0.0)]  # (hypothesis, log_prob)

        # Проходим по каждому времени (T)
        for t in range(log_probs.size(0)):
            all_candidates = []
            # Проходим по текущим гипотезам в лучах
            for hypothesis, score in beams:
                for vocab_idx in range(log_probs.size(1)):
                    token_prob = log_probs[t, vocab_idx]
                    if vocab_idx == self.blank_token_id:
                        # Пропускаем пустой токен
                        continue

                    candidate = (hypothesis + [vocab_idx], score + token_prob.item())
                    all_candidates.append(candidate)

            # Сортируем все кандидаты по вероятности (score) и выбираем top beam_width кандидатов
            beams = heapq.nlargest(self.beam_width, all_candidates, key=lambda x: x[1])

        # Получаем лучшую гипотезу (последний элемент)
        best_hypothesis, _ = beams[0]

        # Преобразуем индексы токенов в строку
        decoded_string = ''.join([self.vocab[idx] for idx in best_hypothesis])

        if return_beams:
            return beams  # Возвращаем все лучи для дальнейшего переоценивания с LM
        else:
            return decoded_string  # Возвращаем лучшую гипотезу


    def lm_rescore(self, beams: List[Tuple[List[int], float]]) -> str:
        """
        Perform second-pass LM rescoring on beam search outputs

        Args:
            beams (list): List of tuples (hypothesis, log_prob)

        Returns:
            str: Best rescored transcript
        """
        if not self.lm_model:
            raise ValueError("KenLM model required for LM rescoring")

        # Список для хранения новых гипотез с переоценёнными вероятностями
        rescored_beams = []

        # Проходим по всем лучам
        for hypothesis, score in beams:
            # Преобразуем индексы гипотезы в строку
            hypothesis_words = [self.vocab[idx] for idx in hypothesis]
            sentence = ' '.join(hypothesis_words)

            # Получаем вероятность для последовательности от модели языка
            lm_prob = self.lm_model.score(sentence)

            # Новый балл гипотезы (суммируем исходную вероятность и вероятность от модели языка)
            combined_score = score + self.alpha * lm_prob  # Мы используем alpha для контроля веса LM

            # Добавляем переоценённую гипотезу в список
            rescored_beams.append((hypothesis, combined_score))

        # Сортируем гипотезы по новым баллам и выбираем лучшую
        rescored_beams = sorted(rescored_beams, key=lambda x: x[1], reverse=True)

        # Получаем лучшую гипотезу (с наибольшим баллом)
        best_hypothesis, _ = rescored_beams[0]

        # Преобразуем индексы токенов в строку
        decoded_string = ''.join([self.vocab[idx] for idx in best_hypothesis])

        return decoded_string

    def decode(self, audio_input: torch.Tensor, method: str = "beam_lm_rescore") -> str:
        """
        Decode input audio file using the specified method

        Args:
            audio_input (torch.Tensor): Audio tensor
            method (str): Decoding method ("greedy", "beam", "beam_lm", "beam_lm_rescore"),
                where "greedy" is a greedy decoding,
                      "beam" is beam search without LM,
                      "beam_lm" is beam search with LM shallow fusion, and
                      "beam_lm_rescore" is a beam search with second pass LM rescoring

        Returns:
            str: Decoded transcription
        """
        # Преобразуем аудиофайл в формат, который подходит для модели
        inputs = self.processor(audio_input, return_tensors="pt", sampling_rate=16000)
        with torch.no_grad():
            logits = self.model(inputs.input_values.squeeze(0)).logits[0]

        # В зависимости от метода декодирования вызываем соответствующую функцию
        if method == "greedy":
            return self.greedy_decode(logits)
        elif method == "beam":
            return self.beam_search_decode(logits)
        elif method == "beam_lm":
            return self.beam_search_with_lm(logits)
        elif method == "beam_lm_rescore":
            beams = self.beam_search_decode(logits, return_beams=True)
            return self.lm_rescore(beams)
        else:
            raise ValueError("Invalid decoding method. Choose one of 'greedy', 'beam', 'beam_lm', 'beam_lm_rescore'.")


In [ ]:
# Тестирование декодера
def test(decoder, audio_path, true_transcription):
    import Levenshtein

    # Загружаем аудиофайл и проверяем частоту дискретизации
    audio_input, sr = torchaudio.load(audio_path)
    assert sr == 16000, "Sample rate must be 16kHz"

    print("=" * 60)
    print("Target transcription")
    print(true_transcription)

    # Тестируем все методы декодирования
    for d_strategy in ["beam_lm_rescore"]: #"beam", "beam_lm", "beam_lm_rescore"]:
        print("-" * 60)
        print(f"{d_strategy} decoding")
        transcript = decoder.decode(audio_input, method=d_strategy)
        print(f"{transcript}")
        print(f"Character-level Levenshtein distance: {Levenshtein.distance(true_transcription, transcript.strip())}")


if __name__ == "__main__":
    # Пример тестовых аудиофайлов и их транскрипций
    test_samples = [
        ("/content/assignments_assignment2_examples_sample1.wav", "IF YOU ARE GENEROUS HERE IS A FITTING OPPORTUNITY FOR THE EXERCISE OF YOUR MAGNANIMITY IF YOU ARE PROUD HERE AM I YOUR RIVAL READY TO ACKNOWLEDGE MYSELF YOUR DEBTOR FOR AN ACT OF THE MOST NOBLE FORBEARANCE"),
        ("/content/sample2.wav", "AND IF ANY OF THE OTHER COPS HAD PRIVATE RACKETS OF THEIR OWN IZZY WAS UNDOUBTEDLY THE MAN TO FIND IT OUT AND USE THE INFORMATION WITH A BEAT SUCH AS THAT EVEN GOING HALVES AND WITH ALL THE GRAFT TO THE UPPER BRACKETS HE'D STILL BE ABLE TO MAKE HIS PILE IN A MATTER OF MONTHS"),
        ("/content/sample3.wav", "GUESS A MAN GETS USED TO ANYTHING HELL MAYBE I CAN HIRE SOME BUMS TO SIT AROUND AND WHOOP IT UP WHEN THE SHIPS COME IN AND BILL THIS AS A REAL OLD MARTIAN DEN OF SIN"),
        ("/content/sample4.wav", "IT WAS A TUNE THEY HAD ALL HEARD HUNDREDS OF TIMES SO THERE WAS NO DIFFICULTY IN TURNING OUT A PASSABLE IMITATION OF IT TO THE IMPROVISED STRAINS OF I DIDN'T WANT TO DO IT THE PRISONER STRODE FORTH TO FREEDOM"),
        ("/content/sample5.wav", "MARGUERITE TIRED OUT WITH THIS LONG CONFESSION THREW HERSELF BACK ON THE SOFA AND TO STIFLE A SLIGHT COUGH PUT UP HER HANDKERCHIEF TO HER LIPS AND FROM THAT TO HER EYES"),
        ("/content/sample6.wav", "AT THIS TIME ALL PARTICIPANTS ARE IN A LISTEN ONLY MODE"),
        ("/content/sample7.wav", "THE INCREASE WAS MAINLY ATTRIBUTABLE TO THE NET INCREASE IN THE AVERAGE SIZE OF OUR FLEETS"),
        ("/content/sample8.wav", "OPERATING SURPLUS IS A NON CAP FINANCIAL MEASURE WHICH IS DEFINED AS FULLY IN OUR PRESS RELEASE"),
    ]

    # Создаем объект декодера
    decoder = Wav2Vec2Decoder()

    # Тестируем все образцы
    _ = [test(decoder, audio_path, target) for audio_path, target in test_samples]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Target transcription
IF YOU ARE GENEROUS HERE IS A FITTING OPPORTUNITY FOR THE EXERCISE OF YOUR MAGNANIMITY IF YOU ARE PROUD HERE AM I YOUR RIVAL READY TO ACKNOWLEDGE MYSELF YOUR DEBTOR FOR AN ACT OF THE MOST NOBLE FORBEARANCE
------------------------------------------------------------
beam_lm_rescore decoding
NNWWWWWANNWNNINHIIF|||YOOU||WAARE|||||GGHHRRREEENNNEEERRRIALAOUUSSS||||NNNNNWHHHEEREE||IISS||||A||FFEEEITTTTTTIINGG||OOP||PPROORRITTTRRRUUUNNNNIIITTTTRYYY|||FFROOR||||THEE|||NNEEEXXTTTHHERRR||CHLLRRLIIISSSEE|||NNNNNNNNNNNNNNNBHOF|||YYOOURR|||MMMM|AGG|HHNNNAANNNNNNNNIM|MMMIITTTTTTEERRYN||||NNUUUUUUUUUUUUWWUWNWNNNU|UUNIHIIFF||YOOU||WAAREE|||||PPRRRAARRROOUNDD||||||UUNNNNWIIIIIIUIWHHHEEERRE|||HAAMMM|||||HHHHHHHIIIL|||||IIIIUNUIUUNWUNNUUUIYYYYOURR|||RRR|AIIIEVVVVAALLL||||UUUUUWWUWWIWIIIIIWWWRRRHEAD|THHLER||THROU||AC|KKKNNNOWWLLEDDGEE|||MMYYY||||SSSSSU|EEELLFFF|||||TYHOOU|ARR||||||AASSDHOO|EUUPPIIITTTHHHHORRR||||UWUUU||OFFRORR|||AN||||HHRRHACCITT||||OF|||||||MMMOOSSSTT|||SSNKNNNNOOOA

In [ ]:
# c разными параметрами

In [7]:

class Wav2Vec2Decoder:
    def __init__(
            self,
            model_name="facebook/wav2vec2-base-960h",
            lm_model_path="/content/3-gram.pruned.1e-7.arpa",
            beam_width=3,
            alpha=1.0,
            beta=1.0
    ):
        """
        Initialization of Wav2Vec2Decoder class

        Args:
            model_name (str): Pretrained Wav2Vec2 model from transformers
            lm_model_path (str): Path to the KenLM n-gram model (for LM rescoring)
            beam_width (int): Number of hypotheses to keep in beam search
            alpha (float): LM weight for shallow fusion and rescoring
            beta (float): Word bonus for shallow fusion
        """
        self.processor = Wav2Vec2Processor.from_pretrained(model_name)
        self.model = Wav2Vec2ForCTC.from_pretrained(model_name)

        # Словарь модели
        self.vocab = {i: c for c, i in self.processor.tokenizer.get_vocab().items()}
        self.blank_token_id = self.processor.tokenizer.pad_token_id
        self.word_delimiter = self.processor.tokenizer.word_delimiter_token
        self.beam_width = beam_width
        self.alpha = alpha
        self.beta = beta
        self.lm_model = kenlm.Model(lm_model_path) if lm_model_path else None

    def greedy_decode(self, logits: torch.Tensor) -> str:
        # Greedy decode
        pred_ids = torch.argmax(logits, dim=-1)
        pred_tokens = [self.vocab[idx.item()] for idx in pred_ids if idx.item() != self.blank_token_id]
        return ''.join(pred_tokens)

    def beam_search_decode(self, logits: torch.Tensor, return_beams: bool = False):
        log_probs = torch.log_softmax(logits, dim=-1)
        beams = [([], 0.0)]

        for t in range(log_probs.size(0)):
            all_candidates = []
            for hypothesis, score in beams:
                for vocab_idx in range(log_probs.size(1)):
                    token_prob = log_probs[t, vocab_idx]
                    if vocab_idx == self.blank_token_id:
                        continue
                    candidate = (hypothesis + [vocab_idx], score + token_prob.item())
                    all_candidates.append(candidate)

            beams = heapq.nlargest(self.beam_width, all_candidates, key=lambda x: x[1])

        best_hypothesis, _ = beams[0]
        decoded_string = ''.join([self.vocab[idx] for idx in best_hypothesis])

        if return_beams:
            return beams
        else:
            return decoded_string

    def beam_search_with_lm(self, logits: torch.Tensor) -> str:
        log_probs = torch.log_softmax(logits, dim=-1)
        beams = [([], 0.0)]

        for t in range(log_probs.size(0)):
            all_candidates = []
            for hypothesis, score in beams:
                for vocab_idx in range(log_probs.size(1)):
                    token_prob = log_probs[t, vocab_idx]
                    if vocab_idx == self.blank_token_id:
                        continue

                    word = self.vocab[vocab_idx]
                    hypothesis_words = [self.vocab[idx] for idx in hypothesis]
                    sentence = ' '.join(hypothesis_words + [word])

                    lm_prob = self.lm_model.score(sentence) if hypothesis else 0.0
                    combined_score = score + token_prob.item() + self.alpha * lm_prob

                    candidate = (hypothesis + [vocab_idx], combined_score)
                    all_candidates.append(candidate)

            beams = heapq.nlargest(self.beam_width, all_candidates, key=lambda x: x[1])

        best_hypothesis, _ = beams[0]
        decoded_string = ''.join([self.vocab[idx] for idx in best_hypothesis])

        return decoded_string

    def lm_rescore(self, beams: List[Tuple[List[int], float]]) -> str:
        rescored_beams = []

        for hypothesis, score in beams:
            hypothesis_words = [self.vocab[idx] for idx in hypothesis]
            sentence = ' '.join(hypothesis_words)
            lm_prob = self.lm_model.score(sentence)
            combined_score = score + self.alpha * lm_prob
            rescored_beams.append((hypothesis, combined_score))

        rescored_beams = sorted(rescored_beams, key=lambda x: x[1], reverse=True)
        best_hypothesis, _ = rescored_beams[0]
        decoded_string = ''.join([self.vocab[idx] for idx in best_hypothesis])

        return decoded_string

    def decode(self, audio_input: torch.Tensor, method: str = "greedy") -> str:
        inputs = self.processor(audio_input, return_tensors="pt", sampling_rate=16000)
        with torch.no_grad():
            logits = self.model(inputs.input_values.squeeze(0)).logits[0]

        if method == "greedy":
            return self.greedy_decode(logits)
        elif method == "beam":
            return self.beam_search_decode(logits)
        elif method == "beam_lm":
            return self.beam_search_with_lm(logits)
        elif method == "beam_lm_rescore":
            beams = self.beam_search_decode(logits, return_beams=True)
            return self.lm_rescore(beams)
        else:
            raise ValueError("Invalid decoding method. Choose one of 'greedy', 'beam', 'beam_lm', 'beam_lm_rescore'.")

def experiment_with_parameters(decoder, audio_path, true_transcription):
    audio_input, sr = torchaudio.load(audio_path)
    assert sr == 16000, "Sample rate must be 16kHz"

    print("=" * 60)
    print("Target transcription")
    print(true_transcription)

    # Параметры
    beam_width_values = [3, 5, 7]
    alpha_values = [0.1, 0.5, 1.0]
    beta_values = [0.1, 0.5, 1.0]

    for beam_width in beam_width_values:
        for alpha in alpha_values:
            for beta in beta_values:
                decoder.beam_width = beam_width
                decoder.alpha = alpha
                decoder.beta = beta

                print("-" * 60)
                print(f"beam_width: {beam_width}, alpha: {alpha}, beta: {beta}")

                # Test different methods
                for d_strategy in ["greedy", "beam", "beam_lm", "beam_lm_rescore"]:
                    print(f"{d_strategy} decoding")
                    transcript = decoder.decode(audio_input, method=d_strategy)
                    distance = Levenshtein.distance(true_transcription, transcript.strip())
                    print(f"Decoded: {transcript}")
                    print(f"Character-level Levenshtein distance: {distance}")

if __name__ == "__main__":
    # Example audio file and transcription
    test_samples = [
         ("/content/assignments_assignment2_examples_sample1.wav", "IF YOU ARE GENEROUS HERE IS A FITTING OPPORTUNITY FOR THE EXERCISE OF YOUR MAGNANIMITY IF YOU ARE PROUD HERE AM I YOUR RIVAL READY TO ACKNOWLEDGE MYSELF YOUR DEBTOR FOR AN ACT OF THE MOST NOBLE FORBEARANCE"),
        ("/content/sample2.wav", "AND IF ANY OF THE OTHER COPS HAD PRIVATE RACKETS OF THEIR OWN IZZY WAS UNDOUBTEDLY THE MAN TO FIND IT OUT AND USE THE INFORMATION WITH A BEAT SUCH AS THAT EVEN GOING HALVES AND WITH ALL THE GRAFT TO THE UPPER BRACKETS HE'D STILL BE ABLE TO MAKE HIS PILE IN A MATTER OF MONTHS"),
        ("/content/sample3.wav", "GUESS A MAN GETS USED TO ANYTHING HELL MAYBE I CAN HIRE SOME BUMS TO SIT AROUND AND WHOOP IT UP WHEN THE SHIPS COME IN AND BILL THIS AS A REAL OLD MARTIAN DEN OF SIN"),
        ("/content/sample4.wav", "IT WAS A TUNE THEY HAD ALL HEARD HUNDREDS OF TIMES SO THERE WAS NO DIFFICULTY IN TURNING OUT A PASSABLE IMITATION OF IT TO THE IMPROVISED STRAINS OF I DIDN'T WANT TO DO IT THE PRISONER STRODE FORTH TO FREEDOM"),
        ("/content/sample5.wav", "MARGUERITE TIRED OUT WITH THIS LONG CONFESSION THREW HERSELF BACK ON THE SOFA AND TO STIFLE A SLIGHT COUGH PUT UP HER HANDKERCHIEF TO HER LIPS AND FROM THAT TO HER EYES"),
        ("/content/sample6.wav", "AT THIS TIME ALL PARTICIPANTS ARE IN A LISTEN ONLY MODE"),
        ("/content/sample7.wav", "THE INCREASE WAS MAINLY ATTRIBUTABLE TO THE NET INCREASE IN THE AVERAGE SIZE OF OUR FLEETS"),
        ("/content/sample8.wav", "OPERATING SURPLUS IS A NON CAP FINANCIAL MEASURE WHICH IS DEFINED AS FULLY IN OUR PRESS RELEASE"),
    ]

    # Create decoder instance
    decoder = Wav2Vec2Decoder()

    # Run experiments with different parameters
    for audio_path, target in test_samples:
        experiment_with_parameters(decoder, audio_path, target)


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Target transcription
IF YOU ARE GENEROUS HERE IS A FITTING OPPORTUNITY FOR THE EXERCISE OF YOUR MAGNANIMITY IF YOU ARE PROUD HERE AM I YOUR RIVAL READY TO ACKNOWLEDGE MYSELF YOUR DEBTOR FOR AN ACT OF THE MOST NOBLE FORBEARANCE
------------------------------------------------------------
beam_width: 3, alpha: 0.1, beta: 0.1
greedy decoding
Decoded: IF|YOOU|AARE||||GENNERROUSS||HHEREE|ISS||||A|FITTTINGG||OPPPOORRTTUNNITTY||FOOR||||THEE||EXERRCISSE||OF||YOURR||MMAGNNANNIMITY|||IFF||YOOU||ARE|||PRROUD||||HEERRE|AMM|||II|||YOUR||RIVALL|||RRETER|TO|ACKKNNOWLEDDGEE|||MYSSELLF|||||YOURR|||DEPTTORR|||FOR|||AN|||ACCT||OF|||||||MMOSTT|||NOBLLE||FORBEAARRANCE||
Character-level Levenshtein distance: 146
beam decoding
Decoded: NIWWWWWANNWNNINHIIF|||YOOU||WAARE|||||GGHHRRREEENNNEEERRRIALAOUUSSS||||NNNNNWHHHEEREE||IISS||||A||FFEEEITTTTTTIINGG||OOP||PPROORRITTTRRRUUUNNNNIIITTTTRYYY|||FFROOR||||THEE|||NNEEEXXTTTHHERRR||CHLLRRLIIISSSEE|||NNNNNNNNNNNNNNNBHOF|||YYOOURR|||MMMM|AGG|HHNNNAANNNNNNNNIM|MMMIITTT

In [ ]:
# модель 4-gram.arpa

In [13]:
!wget https://www.openslr.org/resources/11/4-gram.arpa.gz



--2025-04-12 17:29:51--  https://www.openslr.org/resources/11/4-gram.arpa.gz
Resolving www.openslr.org (www.openslr.org)... 46.101.158.64
Connecting to www.openslr.org (www.openslr.org)|46.101.158.64|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://openslr.elda.org/resources/11/4-gram.arpa.gz [following]
--2025-04-12 17:29:52--  https://openslr.elda.org/resources/11/4-gram.arpa.gz
Resolving openslr.elda.org (openslr.elda.org)... 141.94.109.138, 2001:41d0:203:ad8a::
Connecting to openslr.elda.org (openslr.elda.org)|141.94.109.138|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1355172078 (1.3G) [application/x-gzip]
Saving to: ‘4-gram.arpa.gz.1’

4-gram.arpa.gz.1    100%[===================>]   1.26G  14.6MB/s    in 79s     

2025-04-12 17:31:11 (16.4 MB/s) - ‘4-gram.arpa.gz.1’ saved [1355172078/1355172078]



In [18]:
!mv 4-gram.arpa.gz.1 4-gram.arpa.gz


In [19]:
!gunzip 4-gram.arpa.gz



gzip: 4-gram.arpa.gz: decompression OK, trailing garbage ignored


In [20]:
!ls /content/

3-gram.pruned.1e-7.arpa			      sample2.wav  sample6.wav
4-gram.arpa				      sample3.wav  sample7.wav
4-gram.arpa.gz				      sample4.wav  sample8.wav
assignments_assignment2_examples_sample1.wav  sample5.wav  sample_data


In [21]:
class Wav2Vec2Decoder:
    def __init__(
            self,
            model_name="facebook/wav2vec2-base-960h",
            lm_model_path="/content/4-gram.arpa",
            beam_width=3,
            alpha=1.0,
            beta=1.0
    ):
        """
        Initialization of Wav2Vec2Decoder class

        Args:
            model_name (str): Pretrained Wav2Vec2 model from transformers
            lm_model_path (str): Path to the KenLM n-gram model (for LM rescoring)
            beam_width (int): Number of hypotheses to keep in beam search
            alpha (float): LM weight for shallow fusion and rescoring
            beta (float): Word bonus for shallow fusion
        """
        self.processor = Wav2Vec2Processor.from_pretrained(model_name)
        self.model = Wav2Vec2ForCTC.from_pretrained(model_name)

        # Словарь модели
        self.vocab = {i: c for c, i in self.processor.tokenizer.get_vocab().items()}
        self.blank_token_id = self.processor.tokenizer.pad_token_id
        self.word_delimiter = self.processor.tokenizer.word_delimiter_token
        self.beam_width = beam_width
        self.alpha = alpha
        self.beta = beta
        self.lm_model = kenlm.Model(lm_model_path) if lm_model_path else None

    def greedy_decode(self, logits: torch.Tensor) -> str:
        # Greedy decode
        pred_ids = torch.argmax(logits, dim=-1)
        pred_tokens = [self.vocab[idx.item()] for idx in pred_ids if idx.item() != self.blank_token_id]
        return ''.join(pred_tokens)

    def beam_search_decode(self, logits: torch.Tensor, return_beams: bool = False):
        log_probs = torch.log_softmax(logits, dim=-1)
        beams = [([], 0.0)]

        for t in range(log_probs.size(0)):
            all_candidates = []
            for hypothesis, score in beams:
                for vocab_idx in range(log_probs.size(1)):
                    token_prob = log_probs[t, vocab_idx]
                    if vocab_idx == self.blank_token_id:
                        continue
                    candidate = (hypothesis + [vocab_idx], score + token_prob.item())
                    all_candidates.append(candidate)

            beams = heapq.nlargest(self.beam_width, all_candidates, key=lambda x: x[1])

        best_hypothesis, _ = beams[0]
        decoded_string = ''.join([self.vocab[idx] for idx in best_hypothesis])

        if return_beams:
            return beams
        else:
            return decoded_string

    def beam_search_with_lm(self, logits: torch.Tensor) -> str:
        log_probs = torch.log_softmax(logits, dim=-1)
        beams = [([], 0.0)]

        for t in range(log_probs.size(0)):
            all_candidates = []
            for hypothesis, score in beams:
                for vocab_idx in range(log_probs.size(1)):
                    token_prob = log_probs[t, vocab_idx]
                    if vocab_idx == self.blank_token_id:
                        continue

                    word = self.vocab[vocab_idx]
                    hypothesis_words = [self.vocab[idx] for idx in hypothesis]
                    sentence = ' '.join(hypothesis_words + [word])

                    lm_prob = self.lm_model.score(sentence) if hypothesis else 0.0
                    combined_score = score + token_prob.item() + self.alpha * lm_prob

                    candidate = (hypothesis + [vocab_idx], combined_score)
                    all_candidates.append(candidate)

            beams = heapq.nlargest(self.beam_width, all_candidates, key=lambda x: x[1])

        best_hypothesis, _ = beams[0]
        decoded_string = ''.join([self.vocab[idx] for idx in best_hypothesis])

        return decoded_string

    def lm_rescore(self, beams: List[Tuple[List[int], float]]) -> str:
        rescored_beams = []

        for hypothesis, score in beams:
            hypothesis_words = [self.vocab[idx] for idx in hypothesis]
            sentence = ' '.join(hypothesis_words)
            lm_prob = self.lm_model.score(sentence)
            combined_score = score + self.alpha * lm_prob
            rescored_beams.append((hypothesis, combined_score))

        rescored_beams = sorted(rescored_beams, key=lambda x: x[1], reverse=True)
        best_hypothesis, _ = rescored_beams[0]
        decoded_string = ''.join([self.vocab[idx] for idx in best_hypothesis])

        return decoded_string

    def decode(self, audio_input: torch.Tensor, method: str = "greedy") -> str:
        inputs = self.processor(audio_input, return_tensors="pt", sampling_rate=16000)
        with torch.no_grad():
            logits = self.model(inputs.input_values.squeeze(0)).logits[0]

        if method == "greedy":
            return self.greedy_decode(logits)
        elif method == "beam":
            return self.beam_search_decode(logits)
        elif method == "beam_lm":
            return self.beam_search_with_lm(logits)
        elif method == "beam_lm_rescore":
            beams = self.beam_search_decode(logits, return_beams=True)
            return self.lm_rescore(beams)
        else:
            raise ValueError("Invalid decoding method. Choose one of 'greedy', 'beam', 'beam_lm', 'beam_lm_rescore'.")

def experiment_with_parameters(decoder, audio_path, true_transcription):
    audio_input, sr = torchaudio.load(audio_path)
    assert sr == 16000, "Sample rate must be 16kHz"

    print("=" * 60)
    print("Target transcription")
    print(true_transcription)

    # Параметры
    beam_width_values = [3, 5, 7]
    alpha_values = [0.1, 0.5, 1.0]
    beta_values = [0.1, 0.5, 1.0]

    for beam_width in beam_width_values:
        for alpha in alpha_values:
            for beta in beta_values:
                decoder.beam_width = beam_width
                decoder.alpha = alpha
                decoder.beta = beta

                print("-" * 60)
                print(f"beam_width: {beam_width}, alpha: {alpha}, beta: {beta}")

                # Test different methods
                for d_strategy in ["greedy", "beam", "beam_lm", "beam_lm_rescore"]:
                    print(f"{d_strategy} decoding")
                    transcript = decoder.decode(audio_input, method=d_strategy)
                    distance = Levenshtein.distance(true_transcription, transcript.strip())
                    print(f"Decoded: {transcript}")
                    print(f"Character-level Levenshtein distance: {distance}")

if __name__ == "__main__":
    # Example audio file and transcription
    test_samples = [
         ("/content/assignments_assignment2_examples_sample1.wav", "IF YOU ARE GENEROUS HERE IS A FITTING OPPORTUNITY FOR THE EXERCISE OF YOUR MAGNANIMITY IF YOU ARE PROUD HERE AM I YOUR RIVAL READY TO ACKNOWLEDGE MYSELF YOUR DEBTOR FOR AN ACT OF THE MOST NOBLE FORBEARANCE"),
        ("/content/sample2.wav", "AND IF ANY OF THE OTHER COPS HAD PRIVATE RACKETS OF THEIR OWN IZZY WAS UNDOUBTEDLY THE MAN TO FIND IT OUT AND USE THE INFORMATION WITH A BEAT SUCH AS THAT EVEN GOING HALVES AND WITH ALL THE GRAFT TO THE UPPER BRACKETS HE'D STILL BE ABLE TO MAKE HIS PILE IN A MATTER OF MONTHS"),
        ("/content/sample3.wav", "GUESS A MAN GETS USED TO ANYTHING HELL MAYBE I CAN HIRE SOME BUMS TO SIT AROUND AND WHOOP IT UP WHEN THE SHIPS COME IN AND BILL THIS AS A REAL OLD MARTIAN DEN OF SIN"),
        ("/content/sample4.wav", "IT WAS A TUNE THEY HAD ALL HEARD HUNDREDS OF TIMES SO THERE WAS NO DIFFICULTY IN TURNING OUT A PASSABLE IMITATION OF IT TO THE IMPROVISED STRAINS OF I DIDN'T WANT TO DO IT THE PRISONER STRODE FORTH TO FREEDOM"),
        ("/content/sample5.wav", "MARGUERITE TIRED OUT WITH THIS LONG CONFESSION THREW HERSELF BACK ON THE SOFA AND TO STIFLE A SLIGHT COUGH PUT UP HER HANDKERCHIEF TO HER LIPS AND FROM THAT TO HER EYES"),
        ("/content/sample6.wav", "AT THIS TIME ALL PARTICIPANTS ARE IN A LISTEN ONLY MODE"),
        ("/content/sample7.wav", "THE INCREASE WAS MAINLY ATTRIBUTABLE TO THE NET INCREASE IN THE AVERAGE SIZE OF OUR FLEETS"),
        ("/content/sample8.wav", "OPERATING SURPLUS IS A NON CAP FINANCIAL MEASURE WHICH IS DEFINED AS FULLY IN OUR PRESS RELEASE"),
    ]

    # Create decoder instance
    decoder = Wav2Vec2Decoder()

    # Run experiments with different parameters
    for audio_path, target in test_samples:
        experiment_with_parameters(decoder, audio_path, target)


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Target transcription
IF YOU ARE GENEROUS HERE IS A FITTING OPPORTUNITY FOR THE EXERCISE OF YOUR MAGNANIMITY IF YOU ARE PROUD HERE AM I YOUR RIVAL READY TO ACKNOWLEDGE MYSELF YOUR DEBTOR FOR AN ACT OF THE MOST NOBLE FORBEARANCE
------------------------------------------------------------
beam_width: 3, alpha: 0.1, beta: 0.1
greedy decoding
Decoded: IF|YOOU|AARE||||GENNERROUSS||HHEREE|ISS||||A|FITTTINGG||OPPPOORRTTUNNITTY||FOOR||||THEE||EXERRCISSE||OF||YOURR||MMAGNNANNIMITY|||IFF||YOOU||ARE|||PRROUD||||HEERRE|AMM|||II|||YOUR||RIVALL|||RRETER|TO|ACKKNNOWLEDDGEE|||MYSSELLF|||||YOURR|||DEPTTORR|||FOR|||AN|||ACCT||OF|||||||MMOSTT|||NOBLLE||FORBEAARRANCE||
Character-level Levenshtein distance: 146
beam decoding
Decoded: NIWWWWWANNWNNINHIIF|||YOOU||WAARE|||||GGHHRRREEENNNEEERRRIALAOUUSSS||||NNNNNWHHHEEREE||IISS||||A||FFEEEITTTTTTIINGG||OOP||PPROORRITTTRRRUUUNNNNIIITTTTRYYY|||FFROOR||||THEE|||NNEEEXXTTTHHERRR||CHLLRRLIIISSSEE|||NNNNNNNNNNNNNNNBHOF|||YYOOURR|||MMMM|AGG|HHNNNAANNNNNNNNIM|MMMIITTT